# Phase 3 — Step 3.3: TFLite Conversion + Quantization

Converts the trained Keras model into two TensorFlow Lite variants:
1. **float32** — full precision, larger file, used as the accuracy baseline
2. **INT8 quantized** — 4x smaller, much faster on mobile, minimal accuracy loss

The INT8 model is what gets bundled into the Flutter app.

**What is quantization?**

Quantization converts the model's weights and activations from 32-bit floating point to 8-bit integers.  This makes the file roughly 4x smaller and the inference 2-4x faster on mobile CPUs, with typically less than 1% accuracy drop for a well-trained model.

The catch is that INT8 cannot represent fractional values the same way float32 does, so a small calibration step is needed using a "representative dataset" — a small sample of real training data.  This tells the converter the typical range of values seen at each layer, so it can choose sensible scaling factors.

## Cell 1 — Imports

In [ ]:
import os
import numpy as np
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
print(f"tf lite     : {tf.lite.__version__ if hasattr(tf.lite, '__version__') else 'builtin'}")

## Cell 2 — Mount Drive and load the Keras model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_PATH = '/content/drive/MyDrive/nsl-checkpoints/best_model.h5'
DATA_DIR   = '/content/drive/MyDrive/nsl-data'
TFLITE_DIR = '/content/drive/MyDrive/nsl-tflite'
os.makedirs(TFLITE_DIR, exist_ok=True)

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

# Total parameter count (for size estimation later)
n_params = model.count_params()
print(f"\nTotal parameters: {n_params:,}")
print(f"Expected float32 size: {n_params * 4 / 1024:.1f} KB")
print(f"Expected int8 size   : {n_params * 1 / 1024:.1f} KB")

**Why these sizes matter:**

- Each float32 parameter is 4 bytes.  ~600K parameters × 4 = ~2.4 MB for the float32 model.
- Each int8 parameter is 1 byte.  ~600K parameters × 1 = ~600 KB for the int8 model.
- Mobile apps have a 50 MB initial install budget on Android (and the user pays download data costs), so smaller is better as long as accuracy holds.

## Cell 3 — Convert to float32 TFLite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float32 = converter.convert()

float32_path = os.path.join(TFLITE_DIR, 'nsl_model.tflite')
with open(float32_path, 'wb') as f:
    f.write(tflite_float32)

float32_kb = len(tflite_float32) / 1024
print(f"Saved: {float32_path}")
print(f"Size : {float32_kb:.1f} KB  ({float32_kb/1024:.2f} MB)")

**What `TFLiteConverter.from_keras_model` does:**

It traces the Keras model graph and converts each layer to its TFLite equivalent.  Most standard layers (Dense, LSTM, BatchNorm, Dropout) convert directly.  Custom Lambda layers sometimes need special handling, but our model only uses standard layers so the conversion is straightforward.

## Cell 4 — Convert to INT8 quantized TFLite

In [ ]:
# Reload the model (TF converter requires a fresh graph)
model = tf.keras.models.load_model(MODEL_PATH)

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable default optimizations (includes quantization)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Provide a representative dataset for calibration
X = np.load(os.path.join(DATA_DIR, 'X.npy'))

# Use 100 random training samples -- enough for calibration,
# small enough to be fast.
np.random.seed(42)
calibration_indices = np.random.choice(len(X), size=100, replace=False)
calibration_data = X[calibration_indices].astype(np.float32)

def representative_dataset_gen():
    for sample in calibration_data:
        # TFLite expects a batch dimension: (1, 30, 1662)
        yield [sample[np.newaxis, :, :]]

converter.representative_dataset = representative_dataset_gen

# Force full INT8 quantization for both inputs and outputs.
# This is the most aggressive size reduction.
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_int8 = converter.convert()

int8_path = os.path.join(TFLITE_DIR, 'nsl_model_int8.tflite')
with open(int8_path, 'wb') as f:
    f.write(tflite_int8)

int8_kb = len(tflite_int8) / 1024
print(f"Saved: {int8_path}")
print(f"Size : {int8_kb:.1f} KB  ({int8_kb/1024:.2f} MB)")
print(f"\nSize reduction: "
      f"{(1 - int8_kb/float32_kb)*100:.1f}% smaller than float32")

**Why a representative dataset is needed:**

INT8 can only represent integers from -128 to 127.  The original model has weights that can be any small floating-point number.  To convert, the converter needs to know the **range** of values each weight and activation actually takes during real inference.

By running a small sample of real training data through the model once, the converter observes the actual ranges and chooses per-tensor scaling factors that map the float range into the int8 range with minimal loss of precision.  This is called **calibration**.

100 random samples is usually enough.  More samples do not significantly improve accuracy.

## Cell 5 — Verify both models load and run

In [ ]:
print("=" * 60)
print("VERIFYING FLOAT32 MODEL")
print("=" * 60)
interpreter_f32 = tf.lite.Interpreter(model_path=float32_path,
                                    num_threads=2)
interpreter_f32.allocate_tensors()
input_details  = interpreter_f32.get_input_details()
output_details = interpreter_f32.get_output_details()
print(f"  Input  shape : {input_details[0]['shape']},  "
      f"dtype: {input_details[0]['dtype'].__name__}")
print(f"  Output shape : {output_details[0]['shape']},  "
      f"dtype: {output_details[0]['dtype'].__name__}")

# Run a single sample to confirm it works end-to-end
sample = X[0:1].astype(np.float32)
interpreter_f32.set_tensor(input_details[0]['index'], sample)
interpreter_f32.invoke()
output_f32 = interpreter_f32.get_tensor(output_details[0]['index'])
print(f"  Sample output: {output_f32[0]}")
print(f"  Predicted class index: {int(output_f32[0].argmax())}")

print("\n" + "=" * 60)
print("VERIFYING INT8 MODEL")
print("=" * 60)
interpreter_i8 = tf.lite.Interpreter(model_path=int8_path,
                                   num_threads=2)
interpreter_i8.allocate_tensors()
input_details  = interpreter_i8.get_input_details()
output_details = interpreter_i8.get_output_details()
print(f"  Input  shape : {input_details[0]['shape']},  "
      f"dtype: {input_details[0]['dtype'].__name__}")
print(f"  Output shape : {output_details[0]['shape']},  "
      f"dtype: {output_details[0]['dtype'].__name__}")

# INT8 inputs require quantization of the input sample.
# Get the input quantization parameters.
input_scale, input_zero_point = input_details[0]['quantization']
print(f"\n  Input scale       : {input_scale}")
print(f"  Input zero point  : {input_zero_point}")

# Quantize: float -> int8
sample_q = (sample / input_scale + input_zero_point).astype(np.int8)
interpreter_i8.set_tensor(input_details[0]['index'], sample_q)
interpreter_i8.invoke()
output_i8 = interpreter_i8.get_tensor(output_details[0]['index'])

# Dequantize output back to float for comparison
out_scale, out_zero_point = output_details[0]['quantization']
output_i8_float = (output_i8.astype(np.float32) - out_zero_point) * out_scale
print(f"  Sample output (dequantized): {output_i8_float[0]}")
print(f"  Predicted class index: {int(output_i8_float[0].argmax())}")

**Why the input needs manual quantization:**

Full INT8 models expect int8 inputs, not float32.  The model stores `input_scale` and `input_zero_point` so you can convert your float landmark values into the int8 range before calling inference.  In the Flutter app, this conversion happens once per frame inside the TFLite service.

## Cell 6 — Accuracy comparison: float32 vs INT8

In [ ]:
from sklearn.model_selection import train_test_split

_, X_tmp, _, y_tmp = train_test_split(X, np.load(
    os.path.join(DATA_DIR, 'y.npy')), test_size=0.30,
    stratify=np.load(os.path.join(DATA_DIR, 'y.npy')), random_state=42)
_, X_test, _, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50,
    stratify=y_tmp, random_state=42)

def run_tflite(interpreter, X, input_details, output_details,
               is_int8):
    preds = []
    in_scale,  in_zp  = input_details[0]['quantization']
    out_scale, out_zp = output_details[0]['quantization']
    for i in range(len(X)):
        sample = X[i:i+1]
        if is_int8:
            sample = (sample / in_scale + in_zp).astype(np.int8)
        else:
            sample = sample.astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        out = interpreter.get_tensor(output_details[0]['index'])
        if is_int8:
            out = (out.astype(np.float32) - out_zp) * out_scale
        preds.append(int(out[0].argmax()))
    return np.array(preds)

print(f"Test set size: {len(X_test)}")

preds_f32 = run_tflite(interpreter_f32, X_test,
                       input_details, output_details, is_int8=False)
acc_f32 = (preds_f32 == y_test).mean()
print(f"float32 TFLite accuracy: {acc_f32:.4f}  ({acc_f32*100:.1f}%)")

input_details_i8  = interpreter_i8.get_input_details()
output_details_i8 = interpreter_i8.get_output_details()
preds_i8 = run_tflite(interpreter_i8, X_test,
                      input_details_i8, output_details_i8, is_int8=True)
acc_i8 = (preds_i8 == y_test).mean()
print(f"INT8 TFLite accuracy   : {acc_i8:.4f}  ({acc_i8*100:.1f}%)")

delta = acc_f32 - acc_i8
print(f"\nAccuracy delta         : {delta:+.4f}  "
      f"({'float32 better' if delta > 0 else 'INT8 better or equal'})")

results = pd.DataFrame() if False else None  # silence linter
import pandas as pd
summary_df = pd.DataFrame([
    {"model": "float32", "size_kb": float32_kb, "test_acc": acc_f32},
    {"model": "int8",    "size_kb": int8_kb,    "test_acc": acc_i8},
])
print("\n", summary_df.to_string(index=False))

summary_df.to_csv(
    os.path.join(TFLITE_DIR, 'quantization_accuracy_diff.txt'),
    sep='\t', index=False)
print(f"\nSaved quantization_accuracy_diff.txt")

**What the accuracy delta tells you:**

- **delta < 1%** — quantization is essentially free.  Use the INT8 model in production.
- **delta 1-3%** — acceptable tradeoff.  Use INT8 unless you have a strong reason not to.
- **delta > 5%** — quantization is hurting accuracy.  Either use float32 or retrain with quantization-aware training.

## Cell 7 — Latency benchmark

In [ ]:
import time

def benchmark(interpreter, X, input_details, output_details,
              is_int8, n_warmup=10, n_runs=100):
    """Measure per-inference latency in milliseconds."""
    in_scale, in_zp = input_details[0]['quantization']

    # Warmup
    for _ in range(n_warmup):
        sample = X[_ % len(X):_ % len(X) + 1]
        if is_int8:
            sample = (sample / in_scale + in_zp).astype(np.int8)
        else:
            sample = sample.astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()

    # Timed runs
    latencies_ms = []
    for i in range(n_runs):
        sample = X[i % len(X):i % len(X) + 1]
        if is_int8:
            sample = (sample / in_scale + in_zp).astype(np.int8)
        else:
            sample = sample.astype(np.float32)
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        latencies_ms.append((time.perf_counter() - t0) * 1000)

    return np.array(latencies_ms)

lat_f32 = benchmark(interpreter_f32, X_test, input_details,
                   output_details, is_int8=False)
lat_i8  = benchmark(interpreter_i8, X_test, input_details_i8,
                   output_details_i8, is_int8=True)

def report(name, lat):
    print(f"\n{name}")
    print(f"  Mean   : {lat.mean():.2f} ms")
    print(f"  Median : {np.median(lat):.2f} ms")
    print(f"  P95    : {np.percentile(lat, 95):.2f} ms")
    print(f"  Min    : {lat.min():.2f} ms")
    print(f"  Max    : {lat.max():.2f} ms")

print("=" * 60)
print("LATENCY BENCHMARK  (Colab CPU, 100 runs after 10 warmup)")
print("=" * 60)
report("float32 TFLite", lat_f32)
report("INT8 TFLite",    lat_i8)
print(f"\nSpeedup from quantization: {lat_f32.mean() / lat_i8.mean():.2f}x")

# Save benchmark to text file
bench_path = '/content/drive/MyDrive/nsl-evaluation/latency_benchmark.txt'
with open(bench_path, 'w') as f:
    f.write("NSL Sign Translator - Latency Benchmark\n")
    f.write("=" * 50 + "\n\n")
    f.write("Hardware: Colab T4 GPU runtime (CPU side tested)\n")
    f.write("Warmup:   10 runs discarded\n")
    f.write("Timed:   100 runs\n\n")
    f.write(f"float32 TFLite:\n")
    f.write(f"  Mean   : {lat_f32.mean():.2f} ms\n")
    f.write(f"  Median : {np.median(lat_f32):.2f} ms\n")
    f.write(f"  P95    : {np.percentile(lat_f32, 95):.2f} ms\n\n")
    f.write(f"INT8 TFLite:\n")
    f.write(f"  Mean   : {lat_i8.mean():.2f} ms\n")
    f.write(f"  Median : {np.median(lat_i8):.2f} ms\n")
    f.write(f"  P95    : {np.percentile(lat_i8, 95):.2f} ms\n\n")
    f.write(f"Speedup: {lat_f32.mean()/lat_i8.mean():.2f}x\n")
print(f"\nSaved {bench_path}")

**How to interpret latency:**

- **< 10 ms** — fast enough for real-time use (30+ FPS frame budget)
- **10-30 ms** — usable for streaming inference (30+ FPS if batched)
- **> 50 ms** — needs optimisation, possibly GPU delegate on device

**Note:** These numbers are from Colab's CPU.  Mobile devices vary widely — a Pixel 7 is much faster than an entry-level Android.  The real test is on your target device.

## Cell 8 — Download both TFLite files

In [ ]:
from google.colab import files

print("Downloading both TFLite models to your laptop...")
files.download(float32_path)
files.download(int8_path)

print("\nSave these files to your repo:")
print(f"  model/tflite/nsl_model.tflite")
print(f"  model/tflite/nsl_model_int8.tflite")

print("\nNext steps:")
print("  1. Save both .tflite files to model/tflite/ in your repo")
print("  2. Copy nsl_model_int8.tflite into flutter_app/assets/models/")
print("  3. Begin Phase 4 - Flutter app integration")